# Figure 6 — Independent Bulk RNA-seq Validation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aistanbulresearch/msep/blob/main/notebooks/figures/figure6_bulk_validation.ipynb)

Reproduces **Figure 6** from Çavuş & Kuşkucu (2026): the single-cell pathway ranking (EMT most homogeneous → immune evasion most heterogeneous) is preserved in an independent bulk RNA-seq cohort (Halvorsen 2023, GSE205457), across platforms, institutions, and measurement scales.

**Panels**

| Panel | What it shows | Source |
|---|---|---|
| A | scRNA-seq CV (Arrieta) vs bulk CV (Halvorsen) per pathway — ranking concordance | paper values + msep.profile |
| B | Chordoma vs notochord pathway CV from GSE205457 | paper values |
| C | Per-patient EMT CV stability across six patients, EMT-first marked | `result.pathway_cv` with patient-stratified profile |
| D | Gene-level CV concordance (scRNA vs bulk), Spearman reported | paper values |

## 1. Install

In [ ]:
!pip install -q --upgrade-strategy only-if-needed 'msep[plotting]>=1.1.0'

## 2. Pathway CV — scRNA vs bulk (Panel A)

Values from §3.7 / Figure 6A of the paper. scRNA values are from CSC_TBXT+ cells across cells; bulk values are CV across 6 chordoma tumours (GSE205457). The rankings match.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pair = pd.DataFrame({
    'pathway':        ['emt',  'housekeeping', 'ferroptosis', 'immune_evasion'],
    'scRNA_csc_cv':   [4.632,  1.743,          5.387,         8.548],
    'bulk_tumor_cv':  [0.784,  0.891,          0.904,         1.113],
})
pair

In [ ]:
fig_a, axes = plt.subplots(1, 2, figsize=(11, 4))
sc_sorted = pair.sort_values('scRNA_csc_cv')
bulk_sorted = pair.sort_values('bulk_tumor_cv')

axes[0].barh(sc_sorted['pathway'], sc_sorted['scRNA_csc_cv'], color='#3498DB', edgecolor='white')
axes[0].set_xlabel('CV across cells')
axes[0].set_title('scRNA-seq CV (CSC_TBXT+, Arrieta 2025)')
axes[0].spines[['top', 'right']].set_visible(False)

axes[1].barh(bulk_sorted['pathway'], bulk_sorted['bulk_tumor_cv'], color='#E67E22', edgecolor='white')
axes[1].set_xlabel('CV across tumours')
axes[1].set_title('Bulk RNA-seq CV (Halvorsen 2023, GSE205457)')
axes[1].spines[['top', 'right']].set_visible(False)

fig_a.suptitle('A · Ranking concordance between platforms and cohorts', y=1.02)
fig_a.tight_layout()
fig_a.savefig('figure6_panelA_concordance.pdf', bbox_inches='tight')
fig_a.savefig('figure6_panelA_concordance.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Chordoma vs notochord (Panel B)

Notochord has lower CV overall (normal differentiated tissue), but the *pathway ranking* — EMT most homogeneous — is preserved, consistent with a developmentally inherited EMT programme.

In [ ]:
noto = pd.DataFrame({
    'pathway':       ['emt', 'ferroptosis', 'immune_evasion', 'housekeeping'],
    'chordoma_bulk': [0.784, 0.904, 1.113, 0.891],
    'notochord':     [0.282, 0.341, 0.412, 0.395],
})

fig_b, ax_b = plt.subplots(figsize=(6, 4))
x = range(len(noto))
w = 0.35
ax_b.bar([i - w/2 for i in x], noto['chordoma_bulk'], w, label='Chordoma (bulk)', color='#C0392B')
ax_b.bar([i + w/2 for i in x], noto['notochord'], w, label='Notochord (n=2)', color='#16A085')
ax_b.set_xticks(list(x))
ax_b.set_xticklabels(noto['pathway'])
ax_b.set_ylabel('CV across samples')
ax_b.set_title('B · Chordoma vs notochord pathway CV')
ax_b.legend()
ax_b.spines[['top', 'right']].set_visible(False)
fig_b.tight_layout()
fig_b.savefig('figure6_panelB_notochord.pdf', bbox_inches='tight')
fig_b.savefig('figure6_panelB_notochord.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Per-patient EMT CV stability (Panel C)

Loops over patients in the bundled demo, computes pathway CV within each patient's CSC cells, and reports the EMT-first pattern. On the real cohort, 4/6 patients show EMT as the most homogeneous pathway (paper §3.7).

In [ ]:
import msep

adata = msep.datasets.load_example(n_cells=500, seed=42, n_patients=6)
per_patient = []
for pid, sub in adata[adata.obs['cell_type'] == 'CSC_TBXT+'].obs.groupby('patient_id', observed=True):
    idx = sub.index
    sub_ad = adata[idx].copy()
    if sub_ad.n_obs < 20:
        continue
    r = msep.profile(sub_ad, pathways='cancer_defense', cell_type_key='cell_type',
                     compute_bootstrap=False, compute_gene_cv=False)
    cv_dict = r.pathway_cv.set_index('pathway')['cv'].to_dict()
    cv_dict['patient_id'] = pid
    per_patient.append(cv_dict)
per_patient_df = pd.DataFrame(per_patient).set_index('patient_id')
per_patient_df = per_patient_df[['emt', 'ferroptosis', 'immune_evasion', 'housekeeping']]
per_patient_df['emt_lowest_resistance'] = per_patient_df[['emt', 'ferroptosis', 'immune_evasion']].idxmin(axis=1) == 'emt'
print(per_patient_df.round(3))
print(f'\nEMT-first patients: {per_patient_df["emt_lowest_resistance"].sum()}/{len(per_patient_df)}')

In [ ]:
fig_c, ax_c = plt.subplots(figsize=(7, 4))
for col, colour in [('emt', '#2ECC71'), ('ferroptosis', '#E74C3C'), ('immune_evasion', '#3498DB')]:
    ax_c.plot(per_patient_df.index.astype(str), per_patient_df[col],
              marker='o', label=col, color=colour, linewidth=2)
for i, pid in enumerate(per_patient_df.index):
    if per_patient_df.loc[pid, 'emt_lowest_resistance']:
        ax_c.annotate('★', (i, per_patient_df.loc[pid, 'emt']),
                      ha='center', fontsize=16, color='#27AE60',
                      textcoords='offset points', xytext=(0, 10))
ax_c.set_xlabel('Patient')
ax_c.set_ylabel('CV (CSC_TBXT+ only)')
ax_c.set_title('C · Per-patient pathway CV — ★ = EMT is the most homogeneous resistance pathway')
ax_c.legend(loc='upper right')
ax_c.spines[['top', 'right']].set_visible(False)
fig_c.tight_layout()
fig_c.savefig('figure6_panelC_per_patient.pdf', bbox_inches='tight')
fig_c.savefig('figure6_panelC_per_patient.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Gene-level CV concordance (Panel D)

From the paper's Figure 6D: Spearman correlation between scRNA-seq gene CV and bulk gene CV across the 23 key defence genes. Five genes specifically called out as most conserved: TBXT (0.620), SMAD3 (0.540), SMAD2 (0.700), SLC7A11 (0.563), VIM (0.699).

In [ ]:
concordance_genes = pd.DataFrame({
    'gene':      ['TBXT', 'VIM', 'FN1', 'KRT18', 'SMAD2', 'SMAD3', 'GPX4',
                  'SLC7A11', 'FTH1', 'HLA-E', 'B2M', 'CD274', 'MICA'],
    'scRNA_cv':  [1.02, 3.12, 1.21, 1.70, 1.83, 2.05, 3.14, 2.88, 2.31,
                  1.73, 1.51, 7.21, 2.46],
    'bulk_cv':   [0.620, 0.699, 0.512, 0.584, 0.700, 0.540, 0.621, 0.563,
                  0.482, 0.455, 0.390, 1.204, 0.615],
})

from scipy.stats import spearmanr
rho, p = spearmanr(concordance_genes['scRNA_cv'], concordance_genes['bulk_cv'])

fig_d, ax_d = plt.subplots(figsize=(6, 5))
ax_d.scatter(concordance_genes['scRNA_cv'], concordance_genes['bulk_cv'],
             s=80, edgecolors='black', linewidths=0.8, color='#8E44AD')
for _, row in concordance_genes.iterrows():
    ax_d.annotate(row['gene'], (row['scRNA_cv'], row['bulk_cv']),
                  textcoords='offset points', xytext=(5, 5), fontsize=8)
ax_d.set_xlabel('scRNA-seq CV (CSC, Arrieta 2025)')
ax_d.set_ylabel('Bulk CV (Halvorsen 2023)')
ax_d.set_title(f'D · Gene-level CV concordance (Spearman ρ = {rho:.2f}, p = {p:.2g})')
ax_d.spines[['top', 'right']].set_visible(False)
fig_d.tight_layout()
fig_d.savefig('figure6_panelD_gene_concordance.pdf', bbox_inches='tight')
fig_d.savefig('figure6_panelD_gene_concordance.png', dpi=300, bbox_inches='tight')
plt.show()

## Data availability

- Bulk cohort GEO accession: `GSE205457` (Halvorsen et al., 2023). Six chordomas + two notochord samples + two controls.
- Single-cell cohort: Arrieta et al. 2025 (see Figure 2 notebook, Section 10).